In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('campeonato-brasileiro-full.csv')

# Padroniza nomes
df = df.rename(columns={
    'mandante_Placar': 'mandante_placar',
    'visitante_Placar': 'visitante_placar',
    'mandante_Estado': 'mandante_estado',
    'visitante_Estado': 'visitante_estado'
})

df['data'] = pd.to_datetime(df['data'], dayfirst=True)
df = df.sort_values('data').reset_index(drop=True)

# Resultado
df['resultado'] = np.where(df['mandante_placar'] > df['visitante_placar'], 0,
                np.where(df['mandante_placar'] < df['visitante_placar'], 2, 1))

# Feature engineering simples e correto (só 4 features que funcionam sempre)
teams = list(set(df['mandante']) | set(df['visitante']))
stats = {t: {'hp':0, 'hw':0, 'hgf':0, 'hga':0, 'ap':0, 'aw':0, 'agf':0, 'aga':0} for t in teams}

feats = []
for _, row in df.iterrows():
    h, a = row['mandante'], row['visitante']
    feats.append({
        'h_win': stats[h]['hw'] / max(stats[h]['hp'], 1),
        'a_win': stats[a]['aw'] / max(stats[a]['ap'], 1),
        'h_gf': stats[h]['hgf'] / max(stats[h]['hp'], 1),
        'a_gf': stats[a]['agf'] / max(stats[a]['ap'], 1),
        'derbi': 1 if row['mandante_estado'] == row['visitante_estado'] else 0
    })
    # atualiza
    stats[h]['hp'] += 1; stats[a]['ap'] += 1
    stats[h]['hgf'] += row['mandante_placar']; stats[h]['hga'] += row['visitante_placar']
    stats[a]['agf'] += row['visitante_placar']; stats[a]['aga'] += row['mandante_placar']
    if row['mandante_placar'] > row['visitante_placar']: stats[h]['hw'] += 1
    elif row['mandante_placar'] < row['visitante_placar']: stats[a]['aw'] += 1

df = pd.concat([df, pd.DataFrame(feats)], axis=1)

# Preenche zero com média global
for c in ['h_win','a_win','h_gf','a_gf']:
    df[c] = df[c].replace(0, df[c].mean())

# Split justo: 85% treino, 15% teste (cronológico)
split = int(len(df) * 0.85)
treino = df.iloc[:split]
teste  = df.iloc[split:]

X_train = treino[['h_win','a_win','h_gf','a_gf','derbi']]
X_test  = teste[['h_win','a_win','h_gf','a_gf','derbi']]
y_train = treino['resultado']
y_test  = teste['resultado']

model = HistGradientBoostingClassifier(max_iter=800, learning_rate=0.05, max_depth=7, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"Acurácia: {accuracy_score(y_test, pred):.4f}")
print(f"Baseline (mandante): {(y_test==0).mean():.4f}")
print(f"Total de jogos no teste: {len(teste)}")

from sklearn.metrics import classification_report
print(classification_report(y_test, pred, target_names=['Vit Mandante', 'Empate', 'Vit Visitante']))

Acurácia: 0.4203
Baseline (mandante): 0.4742
Total de jogos no teste: 1318
               precision    recall  f1-score   support

 Vit Mandante       0.52      0.64      0.57       625
       Empate       0.28      0.20      0.23       355
Vit Visitante       0.28      0.24      0.26       338

     accuracy                           0.42      1318
    macro avg       0.36      0.36      0.36      1318
 weighted avg       0.39      0.42      0.40      1318



In [3]:
# Roda isso logo depois de criar o df com as features
print("Primeiros 10 valores de h_win (deve começar com 0.0 e ir subindo):")
print(df['h_win'].head(20).tolist())

print("\nÚltimos 10 valores de h_win (deve estar entre 0.45 e 0.65):")
print(df['h_win'].tail(20).tolist())

print(f"\nMédia geral de h_win: {df['h_win'].mean():.4f}")
print(f"Quantos jogos têm h_win = 0.0000: { (df['h_win'] == 0).sum() }")

Primeiros 10 valores de h_win (deve começar com 0.0 e ir subindo):
[0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148, 0.5053167516195148]

Últimos 10 valores de h_win (deve estar entre 0.45 e 0.65):
[0.5392156862745098, 0.4457831325301205, 0.4398148148148148, 0.4411764705882353, 0.5761124121779859, 0.31125827814569534, 0.5882352941176471, 0.5351351351351351, 0.49414519906323184, 0.5588235294117647, 0.5818181818181818, 0.29333333333333333, 0.48863636363636365, 0.4713896457765668, 0.4574468085106383, 0.5516431924882629, 0.42424242424242425, 0.422360248447205, 0.5490196078431373, 0.58656330749354]

Média geral de h_win: 0.5118
Quantos jogos têm h_win = 0.0000: 0
